# Week 50, Day 3: Depth-first search

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w050d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
adj = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": ["E"], "E": []}

order, seen = [], set()
def visit(v):
    seen.add(v)
    order.append(v)
    for w in adj[v]:
        if w not in seen:
            visit(w)

visit("A")
print(order)

In [ ]:
def dfs_recursive(adj, start):
    order, seen = [], set()
    def visit(v):
        seen.add(v)
        order.append(v)
        for w in adj[v]:
            if w not in seen:
                visit(w)
    visit(start)
    return order

def dfs_stack(adj, start):
    order, seen, stack = [], set(), [start]
    while stack:
        v = stack.pop()
        if v in seen:
            continue
        seen.add(v)
        order.append(v)
        for w in reversed(adj[v]):
            if w not in seen:
                stack.append(w)
    return order

adj = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": ["E"], "E": []}
print(dfs_recursive(adj, "A"), dfs_stack(adj, "A"))
chain = {i: [i + 1] for i in range(5000)}
chain[5000] = []
print(len(dfs_stack(chain, 0)))

In [ ]:
def times(adj, start):
    clock, disc, fin = [0], {}, {}
    def visit(v):
        clock[0] += 1
        disc[v] = clock[0]
        for w in adj[v]:
            if w not in disc:
                visit(w)
        clock[0] += 1
        fin[v] = clock[0]
    visit(start)
    return {v: (disc[v], fin[v]) for v in disc}

adj = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": ["E"], "E": []}
for v, (d, f) in times(adj, "A").items():
    print(v, d, f, " " * d + "(" + "-" * (f - d - 1) + ")")

In [ ]:
WHITE, GREY, BLACK = 0, 1, 2

def find_cycle(adj):
    colour = {v: WHITE for v in adj}
    parent = {}
    def visit(v):
        colour[v] = GREY
        for w in adj[v]:
            if colour[w] == GREY:                  # a back edge: v -> w closes a cycle
                cycle = [v]
                while cycle[-1] != w:
                    cycle.append(parent[cycle[-1]])
                return cycle[::-1]
            if colour[w] == WHITE:
                parent[w] = v
                found = visit(w)
                if found:
                    return found
        colour[v] = BLACK
        return None
    for v in adj:
        if colour[v] == WHITE:
            found = visit(v)
            if found:
                return found
    return None

diamond = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}
loop = {"A": ["B"], "B": ["C"], "C": ["D", "A"], "D": []}
print(find_cycle(diamond), find_cycle(loop))

In [ ]:
def all_paths(adj, start, goal):
    out, path, on_path = [], [start], {start}
    def extend(v):
        if v == goal:
            out.append(list(path))
            return
        for w in adj[v]:
            if w not in on_path:
                path.append(w)
                on_path.add(w)
                extend(w)
                path.pop()                       # backtrack: let other paths use w
                on_path.discard(w)
    extend(start)
    return out

routes = {"home": ["park", "shop"], "park": ["school", "shop"], "shop": ["school"], "school": []}
for p in all_paths(routes, "home", "school"):
    print(" -> ".join(p))

> **Key idea.** DFS follows one path as far as possible before backing up: recursively (watch the recursion limit) or with an explicit stack (check "seen" when popping; push neighbours in reverse to match the recursive order). It's O(V + E). Discovery and finish times nest like brackets; white, grey and black mark unvisited, on the current path, and finished. A directed graph has a cycle exactly when DFS meets a grey vertex (a back edge); an undirected one when it meets a seen vertex other than the parent. Parent links recover the cycle. Backtracking DFS lists every simple path, removing each vertex from the path as it returns. Use BFS for shortest paths, DFS for structure.

### Exercise: Two ways down

Write `dfs_recursive(adj, start)` and `dfs_stack(adj, start)` returning the visit order; the stack version must give the **same** order as the recursive one, and must handle a 20,000-vertex chain. Then `reachable(adj, start)`: the set of vertices reachable from `start`.

In [ ]:
def dfs_recursive(adj, start):
    ___

def dfs_stack(adj, start):
    ___

def reachable(adj, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_order():\n    \"orders\"\n    adj = {\"A\": [\"B\", \"C\"], \"B\": [\"D\"], \"C\": [\"D\"], \"D\": [\"E\"], \"E\": []}\n    assert dfs_recursive(adj, \"A\") == [\"A\", \"B\", \"D\", \"E\", \"C\"] == dfs_stack(adj, \"A\")\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 9)\n        g = {v: rng.sample([w for w in range(n) if w != v], rng.randint(0, n - 1)) for v in range(n)}\n        assert dfs_stack(g, 0) == dfs_recursive(g, 0) and reachable(g, 0) == set(dfs_recursive(g, 0))\n    chain = {i: [i + 1] for i in range(20_000)}\n    chain[20_000] = []\n    assert len(dfs_stack(chain, 0)) == 20_001 and len(reachable(chain, 0)) == 20_001")

### Exercise: Discovery and finish

Write `times(adj)`: run DFS from every undiscovered vertex in the dict's order, and return a dict from vertex to `(discovery, finish)` with one shared clock counting from 1 (each event adds one). Use an explicit stack: push `(vertex, iterator over its neighbours)` pairs, so each vertex is finished when its iterator runs out. Then `finish_order(adj)`: the vertices in order of finish time.

In [ ]:
def times(adj):
    ___

def finish_order(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef rec_times(adj):\n    clock, disc, fin = [0], {}, {}\n    def visit(v):\n        clock[0] += 1\n        disc[v] = clock[0]\n        for w in adj[v]:\n            if w not in disc:\n                visit(w)\n        clock[0] += 1\n        fin[v] = clock[0]\n    for v in adj:\n        if v not in disc:\n            visit(v)\n    return {v: (disc[v], fin[v]) for v in adj}\ndef test_times():\n    \"times\"\n    adj = {\"A\": [\"B\", \"C\"], \"B\": [\"D\"], \"C\": [\"D\"], \"D\": [\"E\"], \"E\": []}\n    assert times(adj) == {\"A\": (1, 10), \"B\": (2, 7), \"D\": (3, 6), \"E\": (4, 5), \"C\": (8, 9)}\n    assert finish_order(adj) == [\"E\", \"D\", \"B\", \"C\", \"A\"]\n    rng = random.Random(2)\n    for _ in range(100):\n        n = rng.randint(1, 9)\n        g = {v: rng.sample([w for w in range(n) if w != v], rng.randint(0, min(3, n - 1))) for v in range(n)}\n        assert times(g) == rec_times(g)\n    chain = {i: [i + 1] for i in range(20_000)}\n    chain[20_000] = []\n    assert times(chain)[0] == (1, 40_002)")

### Exercise: Cycles in a directed graph

Write `find_cycle(adj)`: one cycle of a directed graph as a list of vertices `[v1, v2, ..., vk]` (with an edge from each to the next and from vk back to v1), or `None` if it's acyclic. Use white, grey and black, with an **explicit stack** (the checks use a 20,000-vertex chain). Then `is_dag(adj)`.

In [ ]:
def find_cycle(adj):
    ___

def is_dag(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef is_cycle(adj, c):\n    return c and len(set(c)) == len(c) and all(c[(i + 1) % len(c)] in adj[c[i]] for i in range(len(c)))\ndef test_cycles():\n    \"cycles\"\n    diamond = {\"A\": [\"B\", \"C\"], \"B\": [\"D\"], \"C\": [\"D\"], \"D\": []}\n    loop = {\"A\": [\"B\"], \"B\": [\"C\"], \"C\": [\"D\", \"A\"], \"D\": []}\n    assert find_cycle(diamond) is None and is_dag(diamond)\n    assert is_cycle(loop, find_cycle(loop)) and not is_dag(loop)\n    assert is_cycle({1: [1]}, find_cycle({1: [1]}))\n    rng = random.Random(3)\n    for _ in range(200):\n        n = rng.randint(1, 7)\n        g = {v: [w for w in range(n) if rng.random() < 0.25] for v in range(n)}\n        c = find_cycle(g)\n        # brute force: is there a cycle?\n        has = False\n        for s in g:\n            stack, seen = list(g[s]), set()\n            while stack:\n                x = stack.pop()\n                if x == s:\n                    has = True\n                    break\n                if x not in seen:\n                    seen.add(x)\n                    stack.extend(g[x])\n            if has:\n                break\n        assert (c is not None) == has and (c is None or is_cycle(g, c))\n    chain = {i: [i + 1] for i in range(20_000)}\n    chain[20_000] = [0]\n    assert len(find_cycle(chain)) == 20_001")

### Exercise: Cycles in an undirected graph

For an undirected graph (each edge listed in both vertices' lists, no repeated edges), write `has_cycle(adj)`, using DFS with parent tracking and an explicit stack of `(vertex, parent)` pairs. Then `is_forest(adj)` (no cycles) and `is_tree(adj)` (no cycles, connected, at least one vertex). Cross-check with counting: a forest with c pieces has exactly V − c edges.

In [ ]:
def has_cycle(adj):
    ___

def is_forest(adj):
    ___

def is_tree(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_undirected():\n    \"undirected cycles\"\n    path = {1: [2], 2: [1, 3], 3: [2]}\n    tri = {1: [2, 3], 2: [1, 3], 3: [1, 2]}\n    assert not has_cycle(path) and is_tree(path) and has_cycle(tri) and not is_forest(tri)\n    two = {1: [2], 2: [1], 3: []}\n    assert is_forest(two) and not is_tree(two) and not is_tree({})\n    rng = random.Random(4)\n    for _ in range(200):\n        n = rng.randint(1, 8)\n        edges = {tuple(sorted(rng.sample(range(n), 2))) for _ in range(rng.randint(0, 8))} if n > 1 else set()\n        g = {v: [] for v in range(n)}\n        for a, b in edges:\n            g[a].append(b)\n            g[b].append(a)\n        comp, seen = 0, set()\n        for v in g:\n            if v not in seen:\n                comp += 1\n                stack = [v]\n                while stack:\n                    x = stack.pop()\n                    if x not in seen:\n                        seen.add(x)\n                        stack.extend(g[x])\n        assert has_cycle(g) == (len(edges) > n - comp)\n        assert is_tree(g) == (comp == 1 and len(edges) == n - 1)")

### Exercise: Every route

Write `all_paths(adj, start, goal)`: every simple path from start to goal as a list of lists, in the order a DFS that tries neighbours in list order finds them. Then `count_paths_dag(adj, start, goal)` for a **DAG**: the number of paths, without listing them, using DFS with a memo dict (Week 16) from vertex to its number of paths to the goal, so it's O(V + E) even when there are millions of paths.

In [ ]:
def all_paths(adj, start, goal):
    ___

def count_paths_dag(adj, start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_paths():\n    \"paths\"\n    routes = {\"home\": [\"park\", \"shop\"], \"park\": [\"school\", \"shop\"], \"shop\": [\"school\"], \"school\": []}\n    assert all_paths(routes, \"home\", \"school\") == [[\"home\", \"park\", \"school\"], [\"home\", \"park\", \"shop\", \"school\"], [\"home\", \"shop\", \"school\"]]\n    assert count_paths_dag(routes, \"home\", \"school\") == 3 and all_paths(routes, \"school\", \"home\") == []\n    assert all_paths({1: [2], 2: [1, 3], 3: []}, 1, 3) == [[1, 2, 3]]\n    ladder = {i: [i + 1, i + 2] for i in range(60)}\n    ladder[60], ladder[61] = [61], []\n    assert count_paths_dag(ladder, 0, 61) == 4052739537881")

### Bug hunt: The edge you came in on

This undirected cycle check treats the edge back to the parent as a cycle, so even a simple path "has a cycle": this prints `True`. Skip the parent so it prints `False`.

In [ ]:
def has_cycle(adj):
    seen = set()
    for s in adj:
        if s in seen:
            continue
        seen.add(s)
        stack = [(s, None)]
        while stack:
            v, parent = stack.pop()
            for w in adj[v]:
                if w in seen:
                    return True
                seen.add(w)
                stack.append((w, v))
    return False

print(has_cycle({1: [2], 2: [1, 3], 3: [2]}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"skip the parent\"\n    expect_output(\"False\")\n    assert has_cycle({1: [2, 3], 2: [1, 3], 3: [1, 2]})")

### Bug hunt: Seen isn't the same as on the path

This directed cycle check reports a cycle whenever it meets a vertex it has **ever** seen, so the diamond A → B → D, A → C → D looks cyclic: this prints `True`. Only a vertex still on the current path (grey) closes a cycle; track that separately so it prints `False`.

In [ ]:
def has_cycle(adj):
    seen = set()
    def visit(v):
        seen.add(v)
        for w in adj[v]:
            if w in seen:
                return True
            if visit(w):
                return True
        return False
    return any(visit(v) for v in adj if v not in seen)

print(has_cycle({"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"grey, not seen\"\n    expect_output(\"False\")\n    assert has_cycle({\"A\": [\"B\"], \"B\": [\"A\"]})")

### Bug hunt: Forgetting to step back

This path finder adds each vertex to `on_path` but never removes it when backtracking, so once a vertex has been used by one path, no later path may use it: this prints `2` instead of `3`. Remove it on the way back so it prints `3`.

In [ ]:
def count_paths(adj, start, goal):
    found, on_path = [0], {start}
    def extend(v):
        if v == goal:
            found[0] += 1
            return
        for w in adj[v]:
            if w not in on_path:
                on_path.add(w)
                extend(w)
    extend(start)
    return found[0]

print(count_paths({"home": ["park", "shop"], "park": ["school", "shop"], "shop": ["school"], "school": []}, "home", "school"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"backtrack\"\n    expect_output(\"3\")")

### Puzzle: DFS with a stack

Arrange the lines into an iterative DFS that matches the recursive order. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Directed graph: 1 -> 2, 1 -> 3, 2 -> 4, 3 -> 4, 4 -> 2.
# DFS from 1 tries neighbours in increasing order.

p = ___   # the DFS visit order from 1
q = ___   # the vertex finished first
r = ___   # True or False: the graph has a cycle
s = ___   # the edge that is a back edge, as a tuple
t = ___   # BFS or DFS: which finds shortest paths in unweighted graphs?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2, 4, 3] and q == 4 and r is True, \"p, q and r\"\n    assert s == (4, 2) and t == \"BFS\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: making a maze

DFS can **make** mazes as well as solve them. The "recursive backtracker" starts with a grid of cells, every wall standing. From a start cell, it repeatedly picks a random unvisited neighbour, knocks down the wall between, and moves there; when every neighbour has been visited, it backs up. Because DFS visits each cell once and each knocked-down wall is a tree edge, the result is a **spanning tree** of the grid: every cell reachable, exactly one route between any two, no loops. (Such a maze is called "perfect".)

Write `make_maze(rows, cols, seed)`: run the backtracker with an **explicit stack** from cell (0, 0), using `random.Random(seed)` and choosing among unvisited neighbours with `rng.choice` on the list ordered up, down, left, right. Return the set of passages, each a `frozenset` of two adjacent cells. Then `draw(rows, cols, passages)`: the maze as text, (2·rows + 1) lines of (2·cols + 1) characters, with `#` for walls and spaces for cells and opened walls. Then `route(rows, cols, passages)`: the cells on the path from (0, 0) to the bottom-right cell, found with BFS. The checks verify the maze is perfect (rows·cols − 1 passages, all connected) on mazes up to 60 × 60.

In [ ]:
import random
from collections import deque

def make_maze(rows, cols, seed):
    ___

def draw(rows, cols, passages):
    ___

def route(rows, cols, passages):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_small():\n    \"a small maze\"\n    p = make_maze(3, 4, seed=1)\n    assert len(p) == 11 and all(len(x) == 2 for x in p)\n    for pair in p:\n        (a, b), (c, d) = sorted(pair)\n        assert abs(a - c) + abs(b - d) == 1, \"only adjacent cells\"\n    pic = draw(3, 4, p)\n    lines = pic.split(\"\\n\")\n    assert len(lines) == 7 and all(len(line) == 9 for line in lines) and lines[0] == \"#\" * 9\n    assert pic.count(\" \") == 12 + 11\n    r = route(3, 4, p)\n    assert r[0] == (0, 0) and r[-1] == (2, 3) and all(frozenset((x, y)) in p for x, y in zip(r, r[1:]))\n    assert make_maze(3, 4, seed=1) == p and make_maze(1, 1, seed=0) == set()\ndef test_perfect():\n    \"perfect mazes\"\n    for R, C, s in [(10, 10, 2), (60, 60, 3), (1, 30, 4), (25, 7, 5)]:\n        p = make_maze(R, C, s)\n        assert len(p) == R * C - 1\n        adj = {}\n        for pair in p:\n            a, b = tuple(pair)\n            adj.setdefault(a, []).append(b)\n            adj.setdefault(b, []).append(a)\n        seen, stack = {(0, 0)}, [(0, 0)]\n        while stack:\n            v = stack.pop()\n            for w in adj.get(v, []):\n                if w not in seen:\n                    seen.add(w)\n                    stack.append(w)\n        assert len(seen) == R * C\n        assert route(R, C, p)[-1] == (R - 1, C - 1)")

## Warm-up

BFS from A would visit B and C before D. DFS goes A, B, D, E, and only then, with nothing left below, backs up all the way to A and takes its other branch, C, whose neighbour D has already been seen.

## Going deep

Depth-first search from s: mark s seen, then for each neighbour not yet seen, search from it, completely, before trying the next neighbour. Recursion makes it three lines, and the call stack does the backing up.

Each vertex is visited once and each edge looked at once from each end, so DFS is **O(V + E)**, like BFS. But recursion has Python's limit of about 1,000 levels, and a long path in a graph means deep recursion. With an **explicit stack** there's no limit:

- Push the start. Repeatedly pop a vertex; if it's already seen, skip it; otherwise mark it, record it, and push its unseen neighbours.
- Pushing neighbours in **reverse** order makes the first neighbour pop first, matching the recursive order.

With this version, a vertex can be pushed more than once (from several neighbours) before it's popped, which is why it checks on popping. That's fine: the stack holds at most O(E) entries. The idea goes back at least to Charles Pierre Trémaux, a 19th-century French engineer whose method for solving mazes by marking passages is a depth-first search; John Hopcroft and Robert Tarjan made it a central tool of graph algorithms in the 1970s.

## Entering and leaving

Each vertex has two moments in a DFS: when it's **discovered** (entered) and when it's **finished** (left, after everything reachable from it has been explored). Recording both with a shared clock gives each vertex an interval, and the intervals nest exactly like brackets: if w is discovered during v's visit, w finishes before v does. Many DFS-based algorithms use these times, including Friday's topological sort (which is just "order by finish time, reversed").

A common way to say the same thing is with three **colours**:

- **white**: not yet discovered;
- **grey**: discovered but not finished (it's on the current path, the recursion stack);
- **black**: finished.

The brackets show it: E's interval sits inside D's, inside B's, inside A's; C's comes after B's closes.

## Cycles

**Directed graphs.** During DFS, an edge v → w where w is **grey** points back to a vertex on the current path: a **back edge**, and it closes a cycle. An edge to a **black** vertex doesn't: w was finished earlier and nothing below it leads back. A directed graph has a cycle exactly when DFS finds a back edge. (Checking only "seen" isn't enough: in a diamond A → B → D, A → C → D, the edge C → D reaches a seen vertex without any cycle.)

**Undirected graphs.** Every edge can be walked both ways, so the edge back to the vertex you just came from doesn't count. An undirected graph has a cycle exactly when DFS meets an already-seen vertex that isn't the current vertex's **parent**.

To return the cycle itself, keep parent links; when a back edge v → w is found, walk the parents from v up to w.

## Every path

BFS finds one shortest path. Sometimes you need **all** the simple paths (no repeated vertex) between two vertices: every route through a small network, every way to arrange a puzzle. DFS with **backtracking** does it (Week 16): add a vertex to the path, explore, then **remove it again** so other paths may use it. There can be exponentially many paths, so this suits small graphs only.

## BFS or DFS?

| question | use |
|---|---|
| shortest path in an unweighted graph | BFS |
| everything within k steps; nearest of many | BFS (multi-source) |
| is there any path? what can be reached? | either (DFS is often shorter to write) |
| cycles, back edges, finish order, topological sort | DFS |
| every path, every arrangement (backtracking) | DFS |
| very deep graphs | BFS, or DFS with an explicit stack |

## Practice

## Quiz

## Daily challenge

## Before you go

Print a maze from today's challenge, then solve it with a pencil the depth-first way: keep a hand on one wall, and when you hit a dead end, walk back. Tomorrow: connected components, splitting a graph into its separate pieces, with DFS, BFS and a new structure, union-find.